"""
DecodeLabs – Data Science Project 1
Advanced EDA & Feature Engineering Pipeline
IPO Architecture: Input → Process → Output
"""

# Libraries and reading dataset

In [1]:
import pandas as pd
import numpy as np
import pandera as pa
from pandera import Column, DataFrameSchema, Check
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("Dataset for Data Analytics - Sheet1.csv", parse_dates=["Date"])

In [3]:
print("ORIGINAL DATASET")
print(f"  Shape        : {df.shape}")
print(f"\nMissing values (%):")
miss_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(miss_pct[miss_pct > 0].to_string())

ORIGINAL DATASET
  Shape        : (1200, 14)

Missing values (%):
CouponCode    25.75


# MODULE 1 – INPUT: Securing Fidelity

In [4]:
print("MODULE 1 – MISSING DATA & OUTLIER HANDLING")
 
# Missing Data Decision Matrix 
# CouponCode: 25.75% missing → >20% threshold → KNN Imputation
 
print("\n[Imputation] CouponCode (25.75% missing) → KNN (>20% rule)")
 
le = LabelEncoder()
df["CouponCode_encoded"] = le.fit_transform(df["CouponCode"].fillna("MISSING"))
 
imputer = KNNImputer(n_neighbors=5)
df["CouponCode_encoded"] = imputer.fit_transform(df[["CouponCode_encoded"]])
 
df["CouponCode_encoded"] = (
    df["CouponCode_encoded"]
    .round()
    .astype(int)
    .clip(0, len(le.classes_) - 1)
)
df["CouponCode"] = le.inverse_transform(df["CouponCode_encoded"])
df.drop(columns=["CouponCode_encoded"], inplace=True)
 
print(f"  Remaining nulls: {df['CouponCode'].isnull().sum()}")
print(f"  CouponCode values: {df['CouponCode'].unique()}")

MODULE 1 – MISSING DATA & OUTLIER HANDLING

[Imputation] CouponCode (25.75% missing) → KNN (>20% rule)
  Remaining nulls: 0
  CouponCode values: <StringArray>
['SAVE10', 'FREESHIP', 'MISSING', 'WINTER15']
Length: 4, dtype: str


In [5]:
# Outlier Detection & Winsorization (IQR)

numeric_cols = ["UnitPrice", "TotalPrice", "Quantity", "ItemsInCart"]
 
print("\n[Outlier] IQR Winsorization (numpy.clip) results:")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers_before = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = np.clip(df[col], lower, upper)
    print(f"  {col:15s} | lower={lower:.2f}  upper={upper:.2f} | "f"outliers capped: {outliers_before}")


[Outlier] IQR Winsorization (numpy.clip) results:
  UnitPrice       | lower=-317.20  upper=1024.83 | outliers capped: 0
  TotalPrice      | lower=-1341.41  upper=3330.41 | outliers capped: 8
  Quantity        | lower=-1.00  upper=7.00 | outliers capped: 0
  ItemsInCart     | lower=-0.50  upper=11.50 | outliers capped: 0


# MODULE 2 – PROCESS: Vectorized Computation Engine

In [ ]:
# Feature 1: Revenue Per Item 
df["revenue_per_item"] = df["TotalPrice"] / df["Quantity"]
print("\n[Feature 1] revenue_per_item = TotalPrice / Quantity")


[Feature 1] revenue_per_item = TotalPrice / Quantity


In [ ]:
# Feature 2: Price-to-Cart Ratio 
df["price_to_cart_ratio"] = df["UnitPrice"] / df["ItemsInCart"]
print("[Feature 2] price_to_cart_ratio = UnitPrice / ItemsInCart")

[Feature 2] price_to_cart_ratio = UnitPrice / ItemsInCart


In [ ]:
# Feature 3: Order Month
df["order_month"] = df["Date"].dt.month.astype("int64")
print("[Feature 3] order_month = month extracted from Date")

[Feature 3] order_month = month extracted from Date


In [ ]:
# Feature 4: Has Coupon (binary flag)
df["has_coupon"] = (~df["CouponCode"].isin(["MISSING", ""])).astype(int)
print("[Feature 4] has_coupon = 1 if valid coupon applied, else 0")

[Feature 4] has_coupon = 1 if valid coupon applied, else 0


In [10]:
# Feature 5: Discount Tier
discount_map = {"SAVE10": 10.0, "WINTER15": 15.0, "FREESHIP": 0.0, "MISSING": 0.0}
df["discount_tier"] = df["CouponCode"].map(discount_map).fillna(0.0)
print("[Feature 5] discount_tier = estimated % discount from coupon")

[Feature 5] discount_tier = estimated % discount from coupon


In [11]:
# One-Hot Encoding
cat_cols = ["Product", "PaymentMethod", "OrderStatus", "ReferralSource"]
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
print(f"\n[Encoding] One-Hot Encoded: {cat_cols}")


[Encoding] One-Hot Encoded: ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource']


In [12]:
# Collinearity Eradication
# Step 1: Build absolute correlation matrix
# Step 2: Isolate upper triangle
# Step 3: Identify pairs > 0.80
# Step 4: Drop weaker target correlation

print("\n[Collinearity] Checking pairs with |corr| > 0.80 ...")
num_features = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[num_features].corr().abs()
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
high_corr_pairs = [
    (c, r) for c in upper_tri.columns
    for r in upper_tri.index if upper_tri[c][r] > 0.80
]
 
if high_corr_pairs:
    dropped = set()
    for col_a, col_b in high_corr_pairs:
        if col_a in dropped or col_b in dropped:
            continue
        corr_a = abs(df[col_a].corr(df["TotalPrice"]))
        corr_b = abs(df[col_b].corr(df["TotalPrice"]))
        to_drop = col_a if corr_a < corr_b else col_b
        df.drop(columns=[to_drop], inplace=True)
        dropped.add(to_drop)
        print(f"  Dropped '{to_drop}' (weaker TotalPrice correlation)")
else:
    print("  No collinear pairs found — feature matrix is clean.")


[Collinearity] Checking pairs with |corr| > 0.80 ...
  Dropped 'revenue_per_item' (weaker TotalPrice correlation)


# MODULE 3 – OUTPUT: Structural Contracts (Pandera)

In [13]:
df["discount_tier"] = df["discount_tier"].astype(float)
 
schema = DataFrameSchema({
    "Quantity":            Column(int,     Check.in_range(1, 5)),
    "UnitPrice":           Column(float,   Check.greater_than(0)),
    "ItemsInCart":         Column(int,     Check.in_range(1, 10)),
    "TotalPrice":          Column(float,   Check.greater_than(0)),
    "price_to_cart_ratio": Column(float,   Check.greater_than(0)),
    "has_coupon":          Column(int,     Check.isin([0, 1])),
    "discount_tier":       Column(float,   Check.in_range(0, 100)),
    "order_month":         Column("int64", Check.in_range(1, 12)),
})

In [ ]:
try:
    schema.validate(df, lazy=True)
    print("All schema contracts passed.")
except pa.errors.SchemaErrors as e:
    print(f"Schema failures:\n{e.failure_cases}")

  ✓ All schema contracts passed.


In [ ]:
# Save feature store
df.to_csv("cleaned_feature_store.csv", index=False)
 
print("PIPELINE COMPLETE")
print(f"  Output shape : {df.shape}")
print(f"  Saved        : cleaned_feature_store.csv")
print(f"\nEngineered features:")
for f in ["price_to_cart_ratio", "order_month", "has_coupon",
          "discount_tier"]:
    print(f"  - {f}")
print("  - revenue_per_item (dropped: collinear with UnitPrice)")

PIPELINE COMPLETE
  Output shape : (1200, 32)
  Saved        : cleaned_feature_store.csv

Engineered features:
  - price_to_cart_ratio
  - order_month
  - has_coupon
  - discount_tier
  - revenue_per_item (dropped: collinear with UnitPrice)
